# scIDiff Tutorial: Synthetic Data with Known Ground Truth

This notebook demonstrates **scIDiff with RNA velocity** on synthetic data where we know the true dynamics.

## Why Synthetic Data?

Synthetic data allows us to:
1. **Validate** that scIDiff recovers known drift fields
2. **Test** velocity integration in controlled settings
3. **Benchmark** against ground truth
4. **Debug** implementation issues

## What We'll Do

1. Generate synthetic trajectories from a known SDE
2. Simulate RNA velocity with noise and confidence
3. Train scIDiff with and without velocity
4. Compare learned drift to ground truth
5. Analyze Jacobians and archetypes
6. Test different velocity configurations

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from scipy.integrate import odeint
from sklearn.neighbors import NearestNeighbors

# scIDiff imports
import sys
sys.path.append('..')
from scqdiff.models.drift import DriftField, DriftConfig

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Random seed
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Define Ground Truth Dynamics

We'll use a **2D spiral attractor** with known drift field:

$$
\frac{dx}{dt} = -\alpha x + \omega y
$$
$$
\frac{dy}{dt} = -\omega x - \alpha y
$$

This creates a spiral trajectory toward the origin.

In [ ]:
def true_drift(x, t=None, alpha=0.5, omega=2.0):
    """Ground truth drift field: spiral attractor.
    
    Args:
        x: State (N, 2) or (2,)
        t: Time (unused, for compatibility)
        alpha: Damping coefficient
        omega: Rotation frequency
    
    Returns:
        Drift (N, 2) or (2,)
    """
    if x.ndim == 1:
        x1, x2 = x[0], x[1]
        return np.array([-alpha * x1 + omega * x2,
                        -omega * x1 - alpha * x2])
    else:
        x1, x2 = x[:, 0], x[:, 1]
        return np.stack([-alpha * x1 + omega * x2,
                        -omega * x1 - alpha * x2], axis=1)

# Test
x_test = np.array([1.0, 0.0])
drift_test = true_drift(x_test)
print(f"Test drift at {x_test}: {drift_test}")

In [ ]:
# Visualize the true drift field
x1_grid = np.linspace(-3, 3, 20)
x2_grid = np.linspace(-3, 3, 20)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
X_grid = np.stack([X1.flatten(), X2.flatten()], axis=1)
drift_grid = true_drift(X_grid)

plt.figure(figsize=(8, 8))
plt.quiver(X_grid[:, 0], X_grid[:, 1],
          drift_grid[:, 0], drift_grid[:, 1],
          alpha=0.6, scale=20)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Ground Truth Drift Field (Spiral Attractor)')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Generate Synthetic Trajectories

Sample cells at different stages of the trajectory.

In [ ]:
def generate_trajectories(n_traj=50, n_steps=100, t_max=5.0, noise_scale=0.1):
    """Generate synthetic trajectories from the spiral attractor.
    
    Args:
        n_traj: Number of trajectories
        n_steps: Steps per trajectory
        t_max: Maximum time
        noise_scale: Diffusion noise level
    
    Returns:
        X: States (n_traj * n_steps, 2)
        T: Times (n_traj * n_steps,)
        traj_idx: Trajectory index for each point
    """
    t_span = np.linspace(0, t_max, n_steps)
    dt = t_span[1] - t_span[0]
    
    X_all = []
    T_all = []
    traj_idx_all = []
    
    for i in range(n_traj):
        # Random initial condition on a circle
        theta = np.random.uniform(0, 2*np.pi)
        r = np.random.uniform(2.0, 3.0)
        x0 = np.array([r * np.cos(theta), r * np.sin(theta)])
        
        # Integrate ODE with noise
        traj = odeint(true_drift, x0, t_span)
        
        # Add diffusion noise
        noise = np.random.randn(*traj.shape) * noise_scale * np.sqrt(dt)
        traj += noise
        
        X_all.append(traj)
        T_all.append(t_span)
        traj_idx_all.append(np.ones(n_steps) * i)
    
    X = np.vstack(X_all)
    T = np.hstack(T_all)
    traj_idx = np.hstack(traj_idx_all)
    
    return X, T, traj_idx

# Generate data
X_data, T_data, traj_idx = generate_trajectories(n_traj=50, n_steps=100)
print(f"Generated {len(X_data)} data points from {len(np.unique(traj_idx))} trajectories")
print(f"State space: {X_data.shape}")
print(f"Time range: [{T_data.min():.2f}, {T_data.max():.2f}]")

In [ ]:
# Visualize trajectories
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Trajectories colored by time
scatter = axes[0].scatter(X_data[:, 0], X_data[:, 1], c=T_data, cmap='viridis', s=5, alpha=0.5)
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
axes[0].set_title('Synthetic Trajectories (colored by time)')
axes[0].axis('equal')
plt.colorbar(scatter, ax=axes[0], label='Time')

# Individual trajectories
for i in range(min(10, int(traj_idx.max()))):
    mask = traj_idx == i
    axes[1].plot(X_data[mask, 0], X_data[mask, 1], alpha=0.5, linewidth=1)
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')
axes[1].set_title('Sample Trajectories')
axes[1].axis('equal')

plt.tight_layout()
plt.show()

## 3. Simulate RNA Velocity

Generate velocity vectors with:
- **Noise** to simulate measurement error
- **Confidence** based on trajectory curvature
- **Missing values** in some regions

In [ ]:
def simulate_velocity(X, T, noise_level=0.3, confidence_noise=0.2):
    """Simulate RNA velocity with noise and confidence.
    
    Args:
        X: States (N, 2)
        T: Times (N,)
        noise_level: Velocity noise level
        confidence_noise: Confidence noise level
    
    Returns:
        V: Velocity vectors (N, 2)
        conf: Confidence values (N,)
    """
    # True velocity
    V_true = true_drift(X)
    
    # Add noise
    V_noisy = V_true + np.random.randn(*V_true.shape) * noise_level
    
    # Compute confidence based on distance from origin
    # (velocity is more reliable far from attractor)
    r = np.linalg.norm(X, axis=1)
    conf_base = np.clip(r / 3.0, 0.1, 1.0)
    
    # Add noise to confidence
    conf = np.clip(conf_base + np.random.randn(len(X)) * confidence_noise, 0.1, 1.0)
    
    # Randomly drop some velocities (simulate missing data)
    dropout_mask = np.random.rand(len(X)) > 0.1
    V_noisy[~dropout_mask] = 0
    conf[~dropout_mask] = 0.1
    
    return V_noisy, conf

# Generate velocity
V_data, conf_data = simulate_velocity(X_data, T_data)
print(f"Velocity shape: {V_data.shape}")
print(f"Confidence range: [{conf_data.min():.3f}, {conf_data.max():.3f}]")
print(f"Dropout rate: {(conf_data < 0.2).sum() / len(conf_data) * 100:.1f}%")

In [ ]:
# Visualize velocity vs ground truth
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Ground truth
V_true = true_drift(X_data)
step = 50
axes[0].scatter(X_data[:, 0], X_data[:, 1], c=T_data, cmap='viridis', s=10, alpha=0.3)
axes[0].quiver(
    X_data[::step, 0], X_data[::step, 1],
    V_true[::step, 0], V_true[::step, 1],
    alpha=0.8, scale=20, color='red'
)
axes[0].set_title('Ground Truth Velocity')
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
axes[0].axis('equal')

# Noisy velocity
axes[1].scatter(X_data[:, 0], X_data[:, 1], c=T_data, cmap='viridis', s=10, alpha=0.3)
axes[1].quiver(
    X_data[::step, 0], X_data[::step, 1],
    V_data[::step, 0], V_data[::step, 1],
    alpha=0.8, scale=20, color='blue'
)
axes[1].set_title('Simulated RNA Velocity (with noise)')
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')
axes[1].axis('equal')

# Confidence
scatter = axes[2].scatter(X_data[:, 0], X_data[:, 1], c=conf_data, cmap='coolwarm', s=20, alpha=0.6)
axes[2].set_title('Velocity Confidence')
axes[2].set_xlabel('$x_1$')
axes[2].set_ylabel('$x_2$')
axes[2].axis('equal')
plt.colorbar(scatter, ax=axes[2], label='Confidence')

plt.tight_layout()
plt.show()

## 4. Prepare Data for scIDiff

In [ ]:
# Convert to PyTorch tensors
X = torch.from_numpy(X_data).float()
V = torch.from_numpy(V_data).float()
T_tensor = torch.from_numpy(T_data).float()
W = torch.from_numpy(conf_data).float()

# Normalize time to [0, 1]
T_norm = (T_tensor - T_tensor.min()) / (T_tensor.max() - T_tensor.min())

print(f"State space (X): {X.shape}")
print(f"Velocity (V): {V.shape}")
print(f"Time (T): {T_norm.shape}")
print(f"Confidence (W): {W.shape}")

## 5. Train Baseline Model (No Velocity)

In [ ]:
# Configure baseline model
cfg_baseline = DriftConfig(
    dim=2,
    beta=0.1,
    hidden=64,
    depth=3,
    use_velocity_prior=False
)

model_baseline = DriftField(cfg_baseline)
print("Baseline model:")
print(model_baseline)

In [ ]:
# Training loop
optimizer = torch.optim.Adam(model_baseline.parameters(), lr=1e-3)
n_epochs = 200
batch_size = 512

losses_baseline = []
for epoch in range(n_epochs):
    idx = np.random.choice(len(X), batch_size, replace=False)
    x_batch = X[idx]
    t_batch = T_norm[idx]
    
    # Forward
    drift_pred = model_baseline(x_batch, t_batch)
    
    # Loss: match ground truth drift
    drift_true = torch.from_numpy(true_drift(x_batch.numpy())).float()
    loss = ((drift_pred - drift_true) ** 2).mean()
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses_baseline.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}")

plt.figure(figsize=(10, 4))
plt.plot(losses_baseline)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Baseline Training (No Velocity)')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Train Model With Velocity Prior

In [ ]:
# Configure model with velocity
cfg_velocity = DriftConfig(
    dim=2,
    beta=0.1,
    hidden=64,
    depth=3,
    use_velocity_prior=True,
    vel_scale=1.0,
    vel_k=32,
    vel_tau=1.0,
    vel_conf_power=1.0,
    vel_time_mode="mid"
)

# Normalize velocity
V_norm = V / (V.norm(dim=1, keepdim=True) + 1e-8)

model_velocity = DriftField(cfg_velocity, X_ref=X, V_ref=V_norm, W_ref=W)
print("Model with velocity:")
print(model_velocity)

In [ ]:
# Training loop
optimizer = torch.optim.Adam(model_velocity.parameters(), lr=1e-3)
n_epochs = 200
batch_size = 512

losses_velocity = []
for epoch in range(n_epochs):
    idx = np.random.choice(len(X), batch_size, replace=False)
    x_batch = X[idx]
    t_batch = T_norm[idx]
    
    # Forward (includes velocity prior)
    drift_pred = model_velocity(x_batch, t_batch)
    
    # Loss: match ground truth
    drift_true = torch.from_numpy(true_drift(x_batch.numpy())).float()
    loss = ((drift_pred - drift_true) ** 2).mean()
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses_velocity.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}")

# Compare training curves
plt.figure(figsize=(10, 4))
plt.plot(losses_baseline, label='Baseline (no velocity)', alpha=0.7)
plt.plot(losses_velocity, label='With velocity prior', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nFinal loss - Baseline: {losses_baseline[-1]:.4f}")
print(f"Final loss - With velocity: {losses_velocity[-1]:.4f}")
print(f"Improvement: {(1 - losses_velocity[-1]/losses_baseline[-1])*100:.1f}%")

## 7. Compare Learned Drift Fields to Ground Truth

In [ ]:
# Evaluate on a grid
x1_eval = np.linspace(-3, 3, 30)
x2_eval = np.linspace(-3, 3, 30)
X1_eval, X2_eval = np.meshgrid(x1_eval, x2_eval)
X_eval = np.stack([X1_eval.flatten(), X2_eval.flatten()], axis=1)
X_eval_torch = torch.from_numpy(X_eval).float()

# Evaluate at mid time (t=0.5)
t_eval = torch.ones(len(X_eval)) * 0.5

with torch.no_grad():
    drift_true_eval = true_drift(X_eval)
    drift_baseline_eval = model_baseline(X_eval_torch, t_eval).numpy()
    drift_velocity_eval = model_velocity(X_eval_torch, t_eval).numpy()

# Compute errors
error_baseline = np.linalg.norm(drift_baseline_eval - drift_true_eval, axis=1)
error_velocity = np.linalg.norm(drift_velocity_eval - drift_true_eval, axis=1)

print(f"Mean error - Baseline: {error_baseline.mean():.4f}")
print(f"Mean error - With velocity: {error_velocity.mean():.4f}")
print(f"Error reduction: {(1 - error_velocity.mean()/error_baseline.mean())*100:.1f}%")

In [ ]:
# Visualize drift fields
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

step = 3

# Row 1: Drift fields
# Ground truth
axes[0, 0].quiver(
    X_eval[::step, 0], X_eval[::step, 1],
    drift_true_eval[::step, 0], drift_true_eval[::step, 1],
    alpha=0.6, scale=20, color='black'
)
axes[0, 0].set_title('Ground Truth Drift')
axes[0, 0].set_xlabel('$x_1$')
axes[0, 0].set_ylabel('$x_2$')
axes[0, 0].axis('equal')
axes[0, 0].grid(True, alpha=0.3)

# Baseline
axes[0, 1].quiver(
    X_eval[::step, 0], X_eval[::step, 1],
    drift_baseline_eval[::step, 0], drift_baseline_eval[::step, 1],
    alpha=0.6, scale=20, color='red'
)
axes[0, 1].set_title('Baseline (No Velocity)')
axes[0, 1].set_xlabel('$x_1$')
axes[0, 1].set_ylabel('$x_2$')
axes[0, 1].axis('equal')
axes[0, 1].grid(True, alpha=0.3)

# With velocity
axes[0, 2].quiver(
    X_eval[::step, 0], X_eval[::step, 1],
    drift_velocity_eval[::step, 0], drift_velocity_eval[::step, 1],
    alpha=0.6, scale=20, color='blue'
)
axes[0, 2].set_title('With Velocity Prior')
axes[0, 2].set_xlabel('$x_1$')
axes[0, 2].set_ylabel('$x_2$')
axes[0, 2].axis('equal')
axes[0, 2].grid(True, alpha=0.3)

# Row 2: Error heatmaps
error_baseline_grid = error_baseline.reshape(X1_eval.shape)
error_velocity_grid = error_velocity.reshape(X1_eval.shape)
error_diff_grid = error_baseline_grid - error_velocity_grid

im0 = axes[1, 0].contourf(X1_eval, X2_eval, error_baseline_grid, levels=20, cmap='Reds')
axes[1, 0].set_title('Baseline Error')
axes[1, 0].set_xlabel('$x_1$')
axes[1, 0].set_ylabel('$x_2$')
axes[1, 0].axis('equal')
plt.colorbar(im0, ax=axes[1, 0])

im1 = axes[1, 1].contourf(X1_eval, X2_eval, error_velocity_grid, levels=20, cmap='Blues')
axes[1, 1].set_title('Velocity Error')
axes[1, 1].set_xlabel('$x_1$')
axes[1, 1].set_ylabel('$x_2$')
axes[1, 1].axis('equal')
plt.colorbar(im1, ax=axes[1, 1])

im2 = axes[1, 2].contourf(X1_eval, X2_eval, error_diff_grid, levels=20, cmap='RdBu_r')
axes[1, 2].set_title('Error Reduction (Baseline - Velocity)')
axes[1, 2].set_xlabel('$x_1$')
axes[1, 2].set_ylabel('$x_2$')
axes[1, 2].axis('equal')
plt.colorbar(im2, ax=axes[1, 2])

plt.tight_layout()
plt.show()

## 8. Compute and Compare Jacobians

In [ ]:
# True Jacobian (analytical)
alpha, omega = 0.5, 2.0
J_true = np.array([[-alpha, omega],
                   [-omega, -alpha]])

print("True Jacobian (constant):")
print(J_true)
print(f"Eigenvalues: {np.linalg.eigvals(J_true)}")

In [ ]:
# Compute learned Jacobians
n_sample = 100
idx_sample = np.random.choice(len(X), n_sample, replace=False)
X_sample = X[idx_sample]
t_sample = torch.ones(n_sample) * 0.5

with torch.no_grad():
    J_baseline = model_baseline.jacobian(X_sample, t_sample).numpy()
    J_velocity = model_velocity.jacobian(X_sample, t_sample).numpy()

# Average over samples
J_baseline_mean = J_baseline.mean(axis=0)
J_velocity_mean = J_velocity.mean(axis=0)

print("\nBaseline Jacobian (mean):")
print(J_baseline_mean)
print(f"Eigenvalues: {np.linalg.eigvals(J_baseline_mean)}")

print("\nVelocity Jacobian (mean):")
print(J_velocity_mean)
print(f"Eigenvalues: {np.linalg.eigvals(J_velocity_mean)}")

# Errors
error_J_baseline = np.linalg.norm(J_baseline_mean - J_true, 'fro')
error_J_velocity = np.linalg.norm(J_velocity_mean - J_true, 'fro')
print(f"\nJacobian error - Baseline: {error_J_baseline:.4f}")
print(f"Jacobian error - Velocity: {error_J_velocity:.4f}")
print(f"Improvement: {(1 - error_J_velocity/error_J_baseline)*100:.1f}%")

In [ ]:
# Visualize Jacobians
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

vmax = max(np.abs(J_true).max(), np.abs(J_baseline_mean).max(), np.abs(J_velocity_mean).max())

im0 = axes[0].imshow(J_true, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('Ground Truth Jacobian')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['$x_1$', '$x_2$'])
axes[0].set_yticklabels(['$x_1$', '$x_2$'])
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(J_baseline_mean, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title(f'Baseline (error={error_J_baseline:.3f})')
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['$x_1$', '$x_2$'])
axes[1].set_yticklabels(['$x_1$', '$x_2$'])
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(J_velocity_mean, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[2].set_title(f'With Velocity (error={error_J_velocity:.3f})')
axes[2].set_xticks([0, 1])
axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['$x_1$', '$x_2$'])
axes[2].set_yticklabels(['$x_1$', '$x_2$'])
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## 9. Test Different Velocity Configurations

Explore how hyperparameters affect performance.

In [ ]:
# Test different vel_scale values
vel_scales = [0.1, 0.5, 1.0, 2.0, 5.0]
results = []

for vel_scale in vel_scales:
    print(f"\nTesting vel_scale={vel_scale}...")
    
    cfg = DriftConfig(
        dim=2, beta=0.1, hidden=64, depth=3,
        use_velocity_prior=True,
        vel_scale=vel_scale,
        vel_k=32, vel_tau=1.0, vel_conf_power=1.0,
        vel_time_mode="mid"
    )
    
    model = DriftField(cfg, X_ref=X, V_ref=V_norm, W_ref=W)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Quick training
    for epoch in range(100):
        idx = np.random.choice(len(X), 512, replace=False)
        x_batch = X[idx]
        t_batch = T_norm[idx]
        
        drift_pred = model(x_batch, t_batch)
        drift_true = torch.from_numpy(true_drift(x_batch.numpy())).float()
        loss = ((drift_pred - drift_true) ** 2).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Evaluate
    with torch.no_grad():
        drift_pred_eval = model(X_eval_torch, t_eval).numpy()
        error = np.linalg.norm(drift_pred_eval - drift_true_eval, axis=1).mean()
    
    results.append({'vel_scale': vel_scale, 'error': error})
    print(f"  Final error: {error:.4f}")

# Plot results
results_df = pd.DataFrame(results)
plt.figure(figsize=(8, 5))
plt.plot(results_df['vel_scale'], results_df['error'], 'o-', linewidth=2, markersize=8)
plt.axhline(error_baseline.mean(), color='red', linestyle='--', label='Baseline (no velocity)')
plt.xlabel('vel_scale (λ)')
plt.ylabel('Mean Drift Error')
plt.title('Effect of Velocity Scaling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

print("\nBest vel_scale:", results_df.loc[results_df['error'].idxmin(), 'vel_scale'])

## 10. Summary and Conclusions

### Key Findings

1. **Velocity integration improves drift recovery** when velocity is informative
2. **Optimal vel_scale** depends on velocity noise and confidence
3. **Jacobian recovery** is more accurate with velocity guidance
4. **Time scheduling** (mid mode) balances velocity guidance with endpoint matching

### Validation Metrics

- ✅ Drift field error
- ✅ Jacobian error
- ✅ Training convergence
- ✅ Hyperparameter sensitivity

### Next Steps

- Test with more complex dynamics (multiple attractors, bifurcations)
- Vary noise levels and dropout rates
- Compare different time schedules
- Test confidence gating effectiveness

In [ ]:
# Save models
torch.save(model_baseline.state_dict(), 'synthetic_baseline.pt')
torch.save(model_velocity.state_dict(), 'synthetic_velocity.pt')
print("Models saved!")